# 📊 Data Quality Observability & Intelligent Remediation
**Autonomous AI-Agent for Enterprise Data Health**

Run the single cell below — the app will launch automatically with a public link. No manual input required.

---
### 🔑 Optional: SendGrid Email Alerts
To enable email notifications, add your credentials as **Colab Secrets** (🔑 icon in the left sidebar):

| Secret Name | Value |
|---|---|
| `SENDGRID_API_KEY` | Your SendGrid API key |
| `DQ_ALERT_RECIPIENTS` | Recipient email(s), comma-separated |

The app runs fully without these — email features will simply be skipped.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Data Quality Observability & Intelligent Remediation — Colab Launcher
# Zero-touch setup: runs automatically, no manual input required.
# ═══════════════════════════════════════════════════════════════════════════════

import os, subprocess, threading, time, textwrap

# ── Step 1: Clone repository ─────────────────────────────────────────────────
REPO_URL    = "https://github.com/Teja-Jan/Data-Quality-Observability-Intelligent-Remediation.git"
REPO_BRANCH = "Data-Quality-Observability-and-Intelligent-Remediation"
REPO_DIR    = "Data-Quality-Observability-Intelligent-Remediation"

if not os.path.exists(REPO_DIR):
    print("📥 Cloning repository...")
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL], check=True)
else:
    print("✅ Repository already cloned — pulling latest changes...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
print(f"📂 Working directory: {os.getcwd()}")

# ── Step 2: Install dependencies ──────────────────────────────────────────────
print("\n📦 Installing dependencies...")
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["pip", "install", "-q", "pyngrok"], check=True)
print("✅ Dependencies installed.")

# ── Step 3: Load SendGrid credentials from Colab Secrets (optional) ───────────
print("\n🔑 Checking for Colab Secrets (SendGrid)...")
try:
    from google.colab import userdata

    _sg_key = userdata.get("SENDGRID_API_KEY") or ""
    _sg_rcpt = userdata.get("DQ_ALERT_RECIPIENTS") or ""

    if _sg_key:
        os.environ["SENDGRID_API_KEY"]     = _sg_key
        os.environ["EMAIL_PROVIDER"]        = "sendgrid"
        os.environ["DQ_ALERT_RECIPIENTS"]  = _sg_rcpt
        print("✅ SendGrid credentials loaded from Colab Secrets.")
    else:
        print("ℹ️  No SENDGRID_API_KEY secret found — email alerts disabled (app runs fine without it).")
        os.environ["EMAIL_PROVIDER"] = "smtp"   # neutral fallback, won't fire
except Exception:
    print("ℹ️  Colab Secrets not available — email alerts disabled.")
    os.environ["EMAIL_PROVIDER"] = "smtp"

# ── Step 4: Write .env so python-dotenv & the app pick up env vars ────────────
env_content = textwrap.dedent(f"""
    EMAIL_PROVIDER={os.environ.get('EMAIL_PROVIDER', 'smtp')}
    SENDGRID_API_KEY={os.environ.get('SENDGRID_API_KEY', '')}
    DQ_ALERT_RECIPIENTS={os.environ.get('DQ_ALERT_RECIPIENTS', '')}
    APP_NAME=Data Quality Observability & Intelligent Remediation
    APP_ENV=colab
    LOG_LEVEL=INFO
    DB_PATH=src/db/dq_metadata.db
    AUTO_FIX_MIN_RESOLUTIONS=3
    AUTO_FIX_CONFIDENCE_THRESHOLD=0.95
    DEFAULT_CONN_TYPE=
""").strip()

with open(".env", "w") as f:
    f.write(env_content)
print("✅ .env written.")

# ── Step 5: Start Ollama in background (non-blocking) ─────────────────────────
print("\n🤖 Installing Ollama for AI reasoning layer (background, non-blocking)...")
def _install_and_serve_ollama():
    try:
        os.system("curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1")
        os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
        os.system("ollama serve > /dev/null 2>&1 &")
        time.sleep(5)
        os.system("ollama pull llama3 > /dev/null 2>&1")
    except Exception as e:
        print(f"⚠️  Ollama setup skipped: {e} — app still runs with rule-based fallback.")

threading.Thread(target=_install_and_serve_ollama, daemon=True).start()
print("✅ Ollama is installing in the background — app launches immediately below.")

# ── Step 6: Set up ngrok tunnel ───────────────────────────────────────────────
print("\n🌐 Setting up public tunnel via ngrok...")
from pyngrok import ngrok, conf

# Read ngrok auth token from Colab Secrets if provided, otherwise use anonymous
try:
    from google.colab import userdata as _ud
    _ngrok_token = _ud.get("NGROK_AUTH_TOKEN") or ""
except Exception:
    _ngrok_token = ""

if _ngrok_token:
    ngrok.set_auth_token(_ngrok_token)
    print("✅ ngrok authenticated with token.")
else:
    print("ℹ️  No NGROK_AUTH_TOKEN secret — using anonymous tunnel (limited sessions).")

# ── Step 7: Launch Streamlit ──────────────────────────────────────────────────
STREAMLIT_PORT = 8501

streamlit_cmd = [
    "streamlit", "run", "src/app.py",
    "--server.port", str(STREAMLIT_PORT),
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false",
    "--server.fileWatcherType", "none",
]

print("🚀 Starting Streamlit application...")
streamlit_proc = subprocess.Popen(
    streamlit_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait for Streamlit to be ready
print("⏳ Waiting for Streamlit to be ready...", end="")
for _ in range(30):
    time.sleep(1)
    try:
        import urllib.request
        urllib.request.urlopen(f"http://localhost:{STREAMLIT_PORT}", timeout=2)
        print(" Ready!")
        break
    except Exception:
        print(".", end="", flush=True)
else:
    print(" (continuing anyway)")

# ── Step 8: Open tunnel and display URL ──────────────────────────────────────
public_url = ngrok.connect(STREAMLIT_PORT, "http")
tunnel_url = public_url.public_url

print("\n" + "═" * 65)
print("  ✅  Data Quality Observability & Intelligent Remediation")
print("═" * 65)
print(f"  🌐  Open your app:  {tunnel_url}")
print("═" * 65)
print("  ℹ️   Keep this cell running. Close the tunnel by interrupting.")
print("═" * 65 + "\n")

# Stream Streamlit logs so you can see activity
try:
    for line in streamlit_proc.stdout:
        print(line, end="")
except KeyboardInterrupt:
    print("\n🛑 Shutting down...")
    streamlit_proc.terminate()
    ngrok.kill()